In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
from config import DOSSIER_CP, QUESTIONS, GEN_MODEL, DOSSIER_RESULTATS
from src.extraction_cp.extraction import analyser_cp
from src.retrieval.retrieval import ouvrir_store
from src.generation.generation import repondre

In [2]:

store = ouvrir_store()
questions = json.loads(QUESTIONS.read_text(encoding="utf-8"))

In [3]:

# 1. Chaque CP est analysée une seule fois, comme au dépôt dans l'application
cp_cache = {}
for nom_cp in sorted({q["cp"] for q in questions}):
    texte_cp, _, contrat_id = analyser_cp(DOSSIER_CP / nom_cp)
    cp_cache[nom_cp] = (texte_cp, contrat_id)

In [4]:

# 2. Une réponse par question
resultats = []
for q in questions:
    texte_cp, contrat_id = cp_cache[q["cp"]]
    r = repondre(store, q["question"], texte_cp, contrat_id)
    resultats.append({
        "id": q["id"], "type": q["type"], "question": q["question"],
        "reponse_attendue": q["reponse_attendue"],
        "reponse": r["reponse"],
        "extraits": [f"[{i}] p.{p.metadata['page']} : {p.page_content}" for i, p in enumerate(r["passages"], start=1)],
        "texte_cp": texte_cp,
        "temps": round(r["temps"], 1),
    })
    print(f"{q['id']} ({q['type']:5}) {r['temps']:5.1f} s")


Q01 (CG   )  19.4 s
Q02 (CG   )  17.8 s
Q03 (CG   )  13.7 s
Q04 (CG   )  13.3 s
Q05 (CG   )  12.0 s
Q06 (CG   )  17.6 s
Q07 (CG   )  10.8 s
Q08 (CG   )  13.1 s
Q09 (CG   )  15.3 s
Q10 (CG   )  14.4 s
Q11 (CP   )  10.7 s
Q12 (CP   )  11.6 s
Q13 (CP   )  10.4 s
Q14 (CP   )  11.4 s
Q15 (CP   )  11.1 s
Q16 (CG+CP)  14.2 s
Q17 (CG+CP)  12.0 s
Q18 (CG+CP)  12.9 s
Q19 (CG+CP)  12.4 s
Q20 (CG+CP)  17.6 s
Q21 (CG+CP)  13.4 s
Q22 (HORS )   0.0 s
Q23 (HORS )  10.0 s
Q24 (HORS )  11.2 s
Q25 (HORS )  11.1 s


In [5]:

# 3. Enregistrement, un fichier par modèle
DOSSIER_RESULTATS.mkdir(exist_ok=True)
fichier = DOSSIER_RESULTATS / f"generation_{GEN_MODEL.replace(':', '-')}_v2.json"
fichier.write_text(json.dumps(resultats, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nEnregistré : {fichier.name}")
print(pd.DataFrame(resultats)["temps"].describe())


Enregistré : generation_qwen3-8b_v2.json
count    25.000000
mean     12.696000
std       3.683802
min       0.000000
25%      11.100000
50%      12.400000
75%      14.200000
max      19.400000
Name: temps, dtype: float64


In [6]:
df = pd.DataFrame(resultats)

PHRASES_REFUS = ["je ne trouve pas cette information", "ne sont pas dans notre base"]

df["refus"] = df["reponse"].str.lower().apply(lambda r: any(p in r for p in PHRASES_REFUS))
df["citation_inventee"] = df["reponse"].str.contains("citation inexistante")
df["structure_ok"] = df["reponse"].str.strip().str.startswith("Réponse")
df["cite_cp"] = df["reponse"].str.contains("[CP]", regex=False)

print(df.groupby("type")[["refus", "citation_inventee", "structure_ok", "cite_cp"]].mean().round(2))
df[["id", "type", "refus", "citation_inventee", "structure_ok", "cite_cp", "temps"]]

       refus  citation_inventee  structure_ok  cite_cp
type                                                  
CG       0.0                0.0           0.0     0.00
CG+CP    0.0                0.0           0.0     0.83
CP       0.0                0.0           0.0     1.00
HORS     1.0                0.0           0.0     0.00


,id,type,refus,citation_inventee,structure_ok,cite_cp,temps
0,Q01,CG,False,False,False,False,19.4
1,Q02,CG,False,False,False,False,17.8
2,Q03,CG,False,False,False,False,13.7
3,Q04,CG,False,False,False,False,13.3
4,Q05,CG,False,False,False,False,12.0
5,Q06,CG,False,False,False,False,17.6
6,Q07,CG,False,False,False,False,10.8
7,Q08,CG,False,False,False,False,13.1
8,Q09,CG,False,False,False,False,15.3
9,Q10,CG,False,False,False,False,14.4


In [7]:
for id_q in ["Q23", "Q24", "Q25"]:
    ligne = df[df["id"] == id_q].iloc[0]
    print(f"===== {id_q} : {ligne['question']}")
    print("ATTENDU :", ligne["reponse_attendue"])
    print("OBTENU  :", ligne["reponse"], "\n")

===== Q23 : Mon contrat couvre-t-il les dommages à ma voiture garée dans la rue ?
ATTENDU : Non. Les biens assurés sont ceux qui se trouvent à l'intérieur des bâtiments, et la responsabilité civile exclut les véhicules terrestres à moteur soumis à l'assurance auto. Il faut une assurance auto. Seule exception : le choc d'un véhicule de tiers contre votre bâtiment.
OBTENU  : Je ne trouve pas cette information dans vos documents. 

===== Q24 : Combien coûtera ma cotisation l'année prochaine ?
ATTENDU : L'information n'est pas dans les documents : le système doit le dire (seule la cotisation actuelle, 412 € TTC/an, figure dans les CP).
OBTENU  : Je ne trouve pas cette information dans vos documents. 

===== Q25 : Quel est le meilleur assureur habitation du marché ?
ATTENDU : Question hors périmètre : le système doit refuser de répondre et rester sur le contrat de l'utilisateur.
OBTENU  : Je ne trouve pas cette information dans vos documents. 



In [8]:
GABARIT = ["en une phrase", "ce que prévoient les CP", "ce que disent les CG, avec leurs numéros"]
df["gabarit_recopie"] = df["reponse"].apply(lambda r: any(g in r for g in GABARIT))
print(df.groupby("type")["gabarit_recopie"].mean().round(2))
print("Total :", df["gabarit_recopie"].sum(), "réponses sur", len(df))

type
CG       0.0
CG+CP    0.0
CP       0.0
HORS     0.0
Name: gabarit_recopie, dtype: float64
Total : 0 réponses sur 25


In [9]:
for id_q in df["id"]:
    ligne = df[df["id"] == id_q].iloc[0]
    print(f"===== {id_q} ({ligne['type']}) : {ligne['question']}")
    print("ATTENDU :", ligne["reponse_attendue"])
    print("OBTENU  :", ligne["reponse"], "\n")

===== Q01 (CG) : Dans quel délai dois-je déclarer un vol à mon assureur ?
ATTENDU : Dans les 2 jours ouvrés après en avoir eu connaissance, via l'application Luko. Il faut aussi prévenir immédiatement les autorités, envoyer l'original du dépôt de plainte, et fournir l'état des pertes sous 8 jours (à remettre aussi à la police ou à la gendarmerie). Si la déclaration est en retard et que ce retard cause un préjudice à l'assureur, l'assuré perd son droit à indemnité (déchéance).
OBTENU  : Dans le cas d'un vol, vous devez déclarer le sinistre à votre assureur via l'application Luko **dès que vous en avez connaissance**, et **au plus tard dans les 2 jours ouvrés** [2]. En cas de vol, ce délai est réduit à 2 jours ouvrés.

Sources :
[2] luko_habitation-etendue_cg_2504.pdf, page 67 

===== Q02 (CG) : Que dois-je faire en cas de dégât des eaux dans mon appartement ?
ATTENDU : 1) Limiter les dégâts tout de suite (couper l'eau, protéger les biens). 2) Déclarer le sinistre via l'application Luko 

In [10]:
#le LLM-as-judge

from typing import Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

class Note(BaseModel):
    justification: str = Field(max_length=400,
        description="En 2 phrases : ce qui est juste, puis ce qui manque ou est inventé")
    justesse: Literal[0, 1, 2] = Field(
        description="0 = faux ou hors sujet ; 1 = partiellement juste ; 2 = juste sur les points clés de la réponse attendue")
    fidelite: Literal[0, 1, 2] = Field(
        description="0 = contient des affirmations absentes des documents ; 1 = un détail non vérifiable ; 2 = tout est dans les documents")

JUGE = """Tu évalues la réponse d'un assistant qui explique un contrat d'assurance habitation.

Question : {question}
Réponse attendue (référence vérifiée par un humain) : {attendu}

Documents fournis à l'assistant :
--- Conditions Particulières ---
{cp}
--- Extraits des Conditions Générales ---
{extraits}

Réponse de l'assistant :
{reponse}

Note deux choses séparément :
- la JUSTESSE : la réponse contient-elle les points clés de la réponse attendue ?
- la FIDÉLITÉ : chaque affirmation de la réponse se trouve-t-elle dans les documents fournis ?"""

juge = ChatOllama(model="qwen3:8b", temperature=0, num_ctx=8192, num_predict=500,
                  reasoning=False).with_structured_output(Note, method="json_schema")

notes = []
for r in resultats:
    try:
        n = juge.invoke(JUGE.format(question=r["question"], attendu=r["reponse_attendue"], cp=r["texte_cp"],
                                    extraits="\n\n".join(r["extraits"]) or "(aucun extrait)", reponse=r["reponse"]))
        notes.append({"id": r["id"], **n.model_dump()})
    except Exception as e:
        notes.append({"id": r["id"], "justification": f"ERREUR {type(e).__name__}", "justesse": None, "fidelite": None})
    print(f"{r['id']}  justesse={notes[-1]['justesse']}  fidélité={notes[-1]['fidelite']}")

df = df.drop(columns=["justification", "justesse", "fidelite"], errors="ignore").merge(pd.DataFrame(notes), on="id")
print(df.groupby("type")[["justesse", "fidelite"]].mean().round(2))
df.to_csv(DOSSIER_RESULTATS / "eval_generation_qwen3-8b.csv", index=False)

Q01  justesse=2  fidélité=2
Q02  justesse=2  fidélité=2
Q03  justesse=2  fidélité=2
Q04  justesse=1  fidélité=1
Q05  justesse=1  fidélité=1
Q06  justesse=1  fidélité=1
Q07  justesse=2  fidélité=2
Q08  justesse=2  fidélité=2
Q09  justesse=2  fidélité=1
Q10  justesse=0  fidélité=1
Q11  justesse=1  fidélité=1
Q12  justesse=1  fidélité=1
Q13  justesse=1  fidélité=1
Q14  justesse=1  fidélité=1
Q15  justesse=1  fidélité=1
Q16  justesse=2  fidélité=1
Q17  justesse=1  fidélité=1
Q18  justesse=2  fidélité=2
Q19  justesse=1  fidélité=1
Q20  justesse=1  fidélité=1
Q21  justesse=1  fidélité=1
Q22  justesse=1  fidélité=1
Q23  justesse=2  fidélité=1
Q24  justesse=2  fidélité=2
Q25  justesse=0  fidélité=1
       justesse  fidelite
type                     
CG         1.50      1.50
CG+CP      1.33      1.17
CP         1.00      1.00
HORS       1.25      1.25


In [11]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

class Note(BaseModel):
    justification: str = Field(max_length=400,
        description="En 2 phrases : ce qui est appuyé par les documents, puis ce qui ne l'est pas ou ce qui manque")
    fidelite: Literal[0, 1, 2] = Field(
        description="0 = contient des affirmations absentes des documents ; 1 = un détail non vérifiable ; 2 = tout est dans les documents")
    pertinence: Literal[0, 1, 2] = Field(
        description="0 = ne répond pas à la question ; 1 = répond en partie ou à côté ; 2 = répond directement à la question")

JUGE = """Tu évalues la réponse d'un assistant qui explique un contrat d'assurance habitation.

Documents fournis à l'assistant :
--- Conditions Particulières ---
{cp}
--- Extraits des Conditions Générales ---
{extraits}

Question de l'utilisateur : {question}

Réponse de l'assistant :
{reponse}

Note deux choses séparément :
- la FIDÉLITÉ : chaque affirmation de la réponse se trouve-t-elle dans les documents ci-dessus ?
- la PERTINENCE : la réponse répond-elle à la question posée ?
Si la réponse est un refus (« je ne trouve pas… »), juge si ce refus est justifié par les documents."""

juge = ChatOllama(model="qwen3:8b", temperature=0, num_ctx=8192, num_predict=500,
                  reasoning=False).with_structured_output(Note, method="json_schema")

notes = []
for r in resultats:
    try:
        n = juge.invoke(JUGE.format(cp=r["texte_cp"], extraits="\n\n".join(r["extraits"]) or "(aucun extrait)",
                                    question=r["question"], reponse=r["reponse"]))
        notes.append({"id": r["id"], **n.model_dump()})
    except Exception as e:
        notes.append({"id": r["id"], "justification": f"ERREUR {type(e).__name__}", "fidelite": None, "pertinence": None})
    print(f"{r['id']}  fidélité={notes[-1]['fidelite']}  pertinence={notes[-1]['pertinence']}")

df = df.drop(columns=["justification", "fidelite", "pertinence"], errors="ignore").merge(pd.DataFrame(notes), on="id")
print(df.groupby("type")[["fidelite", "pertinence"]].mean().round(2))
df.to_csv(DOSSIER_RESULTATS / "eval_generation_qwen3-8b.csv", index=False)

Q01  fidélité=1  pertinence=1
Q02  fidélité=1  pertinence=1
Q03  fidélité=1  pertinence=1
Q04  fidélité=1  pertinence=1
Q05  fidélité=1  pertinence=1
Q06  fidélité=1  pertinence=1
Q07  fidélité=1  pertinence=1
Q08  fidélité=1  pertinence=1
Q09  fidélité=1  pertinence=1
Q10  fidélité=1  pertinence=1
Q11  fidélité=1  pertinence=1
Q12  fidélité=1  pertinence=1
Q13  fidélité=1  pertinence=1
Q14  fidélité=1  pertinence=1
Q15  fidélité=1  pertinence=1
Q16  fidélité=2  pertinence=2
Q17  fidélité=1  pertinence=1
Q18  fidélité=1  pertinence=1
Q19  fidélité=2  pertinence=2
Q20  fidélité=1  pertinence=1
Q21  fidélité=1  pertinence=1
Q22  fidélité=0  pertinence=0
Q23  fidélité=1  pertinence=1
Q24  fidélité=1  pertinence=1
Q25  fidélité=1  pertinence=1
       fidelite  pertinence
type                       
CG         1.00        1.00
CG+CP      1.33        1.33
CP         1.00        1.00
HORS       0.75        0.75


In [12]:
print(df.groupby("type")[["fidelite", "pertinence"]].mean().round(2))

       fidelite  pertinence
type                       
CG         1.00        1.00
CG+CP      1.33        1.33
CP         1.00        1.00
HORS       0.75        0.75
